In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install ultralytics opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.1 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 15_seg_weapon_eval.ipynb
#
# Ablación: impacto de la segmentación previa de personas
# sobre el rendimiento del detector de armas (Modelo B).
#
# Hipótesis: enmascarar el fondo antes de pasar el frame al
# detector de armas reduce los falsos positivos, ya que el
# modelo solo busca armas en regiones de personas.
#
# Dos configuraciones evaluadas:
#   CONFIG A — Sin segmentación (baseline, idéntico a notebook 10)
#              weapon_model recibe el frame completo
#   CONFIG B — Con segmentación previa (yolov8s-seg)
#              seg_model detecta personas -> máscara píxel a píxel
#              -> frame enmascarado -> weapon_model
#
# Bloques:
#   BLOQUE A — mAP a nivel de frame  (clips positivos, ambas configs)
#   BLOQUE B — Clasificación clip-level (positivos + negativos)
#   BLOQUE C — FP por categoría negativa
#   BLOQUE D — Guardado de resultados en Drive (CSV + TXT)
#
# Outputs guardados en OUT_DIR:
#   results_config_A.txt / results_config_B.txt
#   clip_results_A.csv   / clip_results_B.csv
# ============================================================

import json
import os
import shutil
import csv
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
from ultralytics import YOLO

print('✅ Imports OK')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Imports OK


In [ ]:
# ============================================================
# CONFIG — ajusta aquí todos los parámetros
# ============================================================

POS_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/handgun_test.txt'
NEG_LIST = '/content/drive/MyDrive/TFM/datasets/videos/Gun_Action_Recognition_Dataset/splits/no_gun_test.txt'
OUT_DIR  = '/content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation'

WEAPON_DET_WEIGHTS = '/content/drive/MyDrive/TFM/experiments/weapon_det/yolov8m_weapons_B_e50_640/weights/best.pt'
SEG_WEIGHTS        = 'yolov8s-seg.pt'   # se descarga automáticamente de Ultralytics

# Inferencia
IMG_SIZE     = 640
CONF_WEAPON  = 0.25
IOU_NMS      = 0.7
CONF_SEG     = 0.4    # confianza para detección de personas en el seg-model
FRAME_STRIDE = 1
MAX_SECONDS  = 15

# Umbral de clasificación clip-level (igual que notebook 10)
DETECTION_THRESHOLD = 5

# IOU thresholds para mAP
IOU_THRESHOLDS = np.arange(0.5, 1.0, 0.05)

Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
print('✅ Config OK')
print(f'   OUT_DIR: {OUT_DIR}')

✅ Config OK
   OUT_DIR: /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation


In [ ]:
# ============================================================
# CARGAR MODELOS
# Copiar pesos del detector a /content/ para evitar
# desconexiones de Drive durante la inferencia larga.
# ============================================================

LOCAL_WEAPON_WEIGHTS = '/content/weapon_best.pt'
if not os.path.exists(LOCAL_WEAPON_WEIGHTS):
    print('Copiando pesos del detector desde Drive...')
    shutil.copy2(WEAPON_DET_WEIGHTS, LOCAL_WEAPON_WEIGHTS)

weapon_model = YOLO(LOCAL_WEAPON_WEIGHTS)
seg_model    = YOLO(SEG_WEIGHTS)   # descarga yolov8s-seg.pt si no existe

print('✅ weapon_model cargado:', LOCAL_WEAPON_WEIGHTS)
print('✅ seg_model cargado:   ', SEG_WEIGHTS)

Copiando pesos del detector desde Drive...
✅ weapon_model cargado: /content/weapon_best.pt
✅ seg_model cargado:    yolov8s-seg.pt


In [ ]:
# ============================================================
# HELPERS
# ============================================================

def iou(boxA, boxB):
    """IoU entre dos boxes [x1,y1,x2,y2]."""
    xA = max(boxA[0], boxB[0]);  yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2]);  yB = min(boxA[3], boxB[3])
    inter = max(0, xB - xA) * max(0, yB - yA)
    if inter == 0:
        return 0.0
    aA = (boxA[2]-boxA[0]) * (boxA[3]-boxA[1])
    aB = (boxB[2]-boxB[0]) * (boxB[3]-boxB[1])
    return inter / (aA + aB - inter)


def load_gt_boxes(label_json_path):
    """
    Devuelve dict {image_id: [(x1,y1,x2,y2), ...]} en formato xyxy.
    El JSON usa formato COCO: bbox = [x, y, w, h].
    """
    with open(label_json_path) as f:
        data = json.load(f)
    gt = defaultdict(list)
    for ann in data['annotations']:
        x, y, w, h = ann['bbox']
        gt[ann['image_id']].append((x, y, x + w, y + h))
    return gt


def build_person_mask(frame, seg_result):
    """
    Construye una máscara binaria píxel a píxel con las regiones
    de personas detectadas por el modelo de segmentación.

    - seg_result: resultado de seg_model.predict() sobre el frame
    - Clase 0 en COCO = person

    Retorna:
      masked_frame : frame con fondo a negro, solo personas visibles
      n_persons    : número de personas detectadas
    """
    h, w = frame.shape[:2]
    combined_mask = np.zeros((h, w), dtype=bool)
    n_persons = 0

    if (seg_result.masks is not None
            and seg_result.boxes is not None
            and len(seg_result.boxes) > 0):

        classes = seg_result.boxes.cls.cpu().numpy().astype(int)
        # Las máscaras de Ultralytics tienen shape (N, H_mask, W_mask)
        # y están redimensionadas al tamaño original del frame.
        masks_data = seg_result.masks.data.cpu().numpy()  # float32 [0,1]

        for idx, cls_id in enumerate(classes):
            if cls_id != 0:   # solo personas (class 0 en COCO)
                continue
            mask_i = masks_data[idx]   # shape (H_mask, W_mask)
            # Redimensionar al tamaño del frame si es necesario
            if mask_i.shape != (h, w):
                mask_i = cv2.resize(mask_i, (w, h), interpolation=cv2.INTER_LINEAR)
            combined_mask |= (mask_i > 0.5)
            n_persons += 1

    masked_frame = np.zeros_like(frame)
    masked_frame[combined_mask] = frame[combined_mask]
    return masked_frame, n_persons


def predict_weapons(frame_input):
    """
    Ejecuta el detector de armas sobre frame_input.
    Retorna lista de (x1,y1,x2,y2,conf).
    """
    r = weapon_model.predict(frame_input, imgsz=IMG_SIZE, conf=CONF_WEAPON,
                              iou=IOU_NMS, verbose=False, device='cuda')[0]
    preds = []
    if r.boxes is not None and len(r.boxes) > 0:
        for b in r.boxes:
            x1, y1, x2, y2 = map(float, b.xyxy[0])
            preds.append((x1, y1, x2, y2, float(b.conf[0])))
    return preds


print('✅ Helpers OK')

✅ Helpers OK


In [ ]:
# ============================================================
# FUNCIÓN PRINCIPAL: procesar un vídeo en ambas configuraciones
# ============================================================

def run_video(video_path, use_seg):
    """
    Procesa el vídeo frame a frame en una de las dos configs.

    Args:
      video_path : Path al vídeo
      use_seg    : bool
                   False -> CONFIG A: frame completo al detector
                   True  -> CONFIG B: frame enmascarado (solo personas)

    Retorna:
      frame_results : list of (frame_idx, preds)
                      frame_idx es 1-based (coincide con image_id del JSON)
                      preds = [(x1,y1,x2,y2,conf), ...]
      max_frames    : int
    """
    cap      = cv2.VideoCapture(str(video_path))
    fps      = cap.get(cv2.CAP_PROP_FPS) or 30
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    max_f    = min(n_frames, int(MAX_SECONDS * fps))

    frame_results = []
    frame_i = 0

    while frame_i < max_f:
        ok, frame = cap.read()
        if not ok:
            break

        if frame_i % FRAME_STRIDE == 0:
            if use_seg:
                # CONFIG B: segmentar personas primero
                seg_r = seg_model.predict(frame, imgsz=IMG_SIZE, conf=CONF_SEG,
                                          classes=[0],   # solo personas
                                          verbose=False, device='cuda')[0]
                frame_input, _ = build_person_mask(frame, seg_r)
            else:
                # CONFIG A: frame original
                frame_input = frame

            preds = predict_weapons(frame_input)
            frame_results.append((frame_i + 1, preds))   # image_id 1-based

        frame_i += 1

    cap.release()
    return frame_results, max_f


print('✅ run_video() OK')

✅ run_video() OK


In [ ]:
# ============================================================
# BLOQUE A — mAP a nivel de frame
# Solo clips positivos. Se evalúan ambas configuraciones
# en el mismo bucle para evitar doble lectura de vídeos.
# ============================================================
print('\n' + '='*60)
print('BLOQUE A — mAP a nivel de frame')
print('='*60)

pos_paths = [Path(l.strip()) for l in Path(POS_LIST).read_text().splitlines() if l.strip()]

# Acumuladores por config
all_tp = {'A': defaultdict(list), 'B': defaultdict(list)}
all_fp = {'A': defaultdict(list), 'B': defaultdict(list)}
total_gt = 0

for vp in pos_paths:
    if not vp.exists():
        print(f'  ❌ No existe: {vp}')
        continue

    label_path = vp.parent / 'label.json'
    if not label_path.exists():
        print(f'  ⚠️  Sin label.json: {vp.parent.name}')
        continue

    clip_id  = vp.parent.name
    local_in = f'/content/{clip_id}_eval.mp4'
    shutil.copy2(str(vp), local_in)

    gt_boxes = load_gt_boxes(label_path)
    total_gt += sum(len(v) for v in gt_boxes.values())

    for cfg, use_seg in [('A', False), ('B', True)]:
        frame_results, _ = run_video(local_in, use_seg=use_seg)

        for (frame_idx, preds) in frame_results:
            gts = gt_boxes.get(frame_idx, [])
            for iou_thr in IOU_THRESHOLDS:
                matched_gt = set()
                for (px1, py1, px2, py2, conf) in sorted(preds, key=lambda x: -x[4]):
                    best_iou, best_j = 0, -1
                    for j, gt in enumerate(gts):
                        if j in matched_gt:
                            continue
                        s = iou((px1, py1, px2, py2), gt)
                        if s > best_iou:
                            best_iou, best_j = s, j
                    if best_iou >= iou_thr and best_j >= 0:
                        all_tp[cfg][iou_thr].append(1)
                        all_fp[cfg][iou_thr].append(0)
                        matched_gt.add(best_j)
                    else:
                        all_tp[cfg][iou_thr].append(0)
                        all_fp[cfg][iou_thr].append(1)

    os.remove(local_in)
    print(f'  ✅ {clip_id}')

# Calcular métricas por config
map_metrics = {}
for cfg in ['A', 'B']:
    aps = {}
    for iou_thr in IOU_THRESHOLDS:
        tp = sum(all_tp[cfg][iou_thr])
        fp = sum(all_fp[cfg][iou_thr])
        fn = total_gt - tp
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
        aps[iou_thr] = {'precision': precision, 'recall': recall}
    map_metrics[cfg] = {
        'mAP50':    aps[0.5]['precision'],
        'mAP5095':  float(np.mean([v['precision'] for v in aps.values()])),
        'prec50':   aps[0.5]['precision'],
        'rec50':    aps[0.5]['recall'],
    }

print(f'\n  Total GT boxes : {total_gt}')
print(f'  {"":20} {"Config A":>12} {"Config B":>12}  Delta')
print('  ' + '-'*52)
for key, label in [("mAP50","mAP@50"), ("mAP5095","mAP@50:95"),
                   ("prec50","Precision@50"), ("rec50","Recall@50")]:
    a = map_metrics['A'][key]
    b = map_metrics['B'][key]
    print(f'  {label:20} {a:>12.4f} {b:>12.4f}  {b-a:+.4f}')


BLOQUE A — mAP a nivel de frame
  ✅ PAH1_C1_P1_V1_HB_3
  ✅ PAH1_C1_P1_V1_HB_4
  ✅ PAH1_C1_P2_V1_HB_1
  ✅ PAH1_C1_P2_V1_HB_3
  ✅ PAH1_C1_P4_V1_HB_1
  ✅ PAH1_C1_P4_V1_HB_2
  ✅ PAH1_C2_P3_V1_HB_1
  ✅ PAH1_C2_P3_V1_HB_3
  ✅ PAH1_C2_P3_V2_HB_2
  ✅ PAH1_C2_P3_V2_HB_3
  ✅ PAH1_C2_P5_V1_HB_1
  ✅ PAH1_C2_P5_V1_HB_4
  ✅ PAH1_C2_P5_V2_HB_3
  ✅ PAH1_C2_P5_V2_HB_4
  ✅ PAH2_C1_P1_V1_HB_1
  ✅ PAH2_C1_P1_V1_HB_4
  ✅ PAH2_C1_P2_V1_HB_3
  ✅ PAH2_C1_P2_V1_HB_4
  ✅ PAH2_C1_P4_V1_HB_1
  ✅ PAH2_C1_P4_V1_HB_3
  ✅ PAH2_C2_P3_V1_HB_1
  ✅ PAH2_C2_P3_V1_HB_4
  ✅ PAH2_C2_P3_V2_HB_2
  ✅ PAH2_C2_P3_V2_HB_4
  ✅ PAH2_C2_P5_V1_HB_3
  ✅ PAH2_C2_P5_V1_HB_4
  ✅ PAH2_C2_P5_V2_HB_2
  ✅ PAH2_C2_P5_V2_HB_4
  ✅ PAH3_C1_P1_V1_HB_1
  ✅ PAH3_C1_P1_V1_HB_4
  ✅ PAH3_C1_P2_V1_HB_1
  ✅ PAH3_C1_P2_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_2
  ✅ PAH3_C1_P4_V1_HB_4
  ✅ PAH3_C2_P3_V1_HB_1
  ✅ PAH3_C2_P3_V1_HB_2
  ✅ PAH3_C2_P3_V2_HB_1
  ✅ PAH3_C2_P3_V2_HB_3
  ✅ PAH3_C2_P5_V1_HB_2
  ✅ PAH3_C2_P5_V1_HB_4
  ✅ PAH3_C2_P5_V2_HB_2
  ✅ PAH3_C2_P5_V2_HB_4
 

In [ ]:
# ============================================================
# BLOQUE B — Clasificación a nivel de clip
# Positivos + negativos, ambas configuraciones
# ============================================================
print('\n' + '='*60)
print(f'BLOQUE B — Clasificación clip-level (umbral={DETECTION_THRESHOLD} frames)')
print('='*60)

neg_paths = [Path(l.strip()) for l in Path(NEG_LIST).read_text().splitlines() if l.strip()]

# Resultados por config: lista de dicts por clip
clip_results = {'A': [], 'B': []}

cat_labels = {
    'N1': 'Walking empty hands',  'N2': 'Jogging',
    'N3': 'Running',              'N4': 'Sneaking empty hands',
    'N5': 'Phone relaxed',        'N6': 'Phone looking',
    'N7': 'Phone both hands',     'N8': 'Phone recording 1h',
    'N9': 'Phone recording 2h',   'N10': 'Water bottle relaxed',
    'N11': 'Drinking',            'N12': 'Holding heavy object',
}


def classify_from_frame_results(frame_results):
    frames_with_gun = sum(1 for (_, preds) in frame_results if len(preds) > 0)
    predicted = 1 if frames_with_gun >= DETECTION_THRESHOLD else 0
    return predicted, frames_with_gun


def process_split(paths, true_label, split_name):
    print(f'\n  Procesando {split_name} ({len(paths)} clips)...')
    for vp in paths:
        if not vp.exists():
            continue
        clip_id  = vp.parent.name
        category = clip_id.split('_')[0] if true_label == 0 else 'POS'
        local_in = '/content/tmp_cls.mp4'
        shutil.copy2(str(vp), local_in)

        for cfg, use_seg in [('A', False), ('B', True)]:
            frame_results, n_total = run_video(local_in, use_seg=use_seg)
            pred, n_det = classify_from_frame_results(frame_results)
            clip_results[cfg].append({
                'clip':     clip_id,
                'true':     true_label,
                'pred':     pred,
                'det_frames':   n_det,
                'total_frames': n_total,
                'category': category,
            })

        # Imprimir resultado de ambas configs en la misma línea
        r_a = clip_results['A'][-1]
        r_b = clip_results['B'][-1]
        if true_label == 1:
            la = '✅TP' if r_a['pred'] == 1 else '❌FN'
            lb = '✅TP' if r_b['pred'] == 1 else '❌FN'
        else:
            la = '❌FP' if r_a['pred'] == 1 else '✅TN'
            lb = '❌FP' if r_b['pred'] == 1 else '✅TN'
        print(f'    {clip_id:<40}  A={la}  B={lb}')

        os.remove(local_in)


process_split(pos_paths, true_label=1, split_name='positivos')
process_split(neg_paths, true_label=0, split_name='negativos')


def compute_metrics(results):
    y_true = np.array([r['true'] for r in results])
    y_pred = np.array([r['pred'] for r in results])
    TP = int(((y_true==1) & (y_pred==1)).sum())
    TN = int(((y_true==0) & (y_pred==0)).sum())
    FP = int(((y_true==0) & (y_pred==1)).sum())
    FN = int(((y_true==1) & (y_pred==0)).sum())
    acc  = (TP+TN) / len(y_true)
    prec = TP / (TP+FP) if (TP+FP) > 0 else 0
    rec  = TP / (TP+FN) if (TP+FN) > 0 else 0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0
    return {'TP':TP,'TN':TN,'FP':FP,'FN':FN,
            'acc':acc,'prec':prec,'rec':rec,'f1':f1}


m = {cfg: compute_metrics(clip_results[cfg]) for cfg in ['A','B']}

print('\n  --- Métricas globales clip-level ---')
print(f'  {"":20} {"Config A":>12} {"Config B":>12}  Delta')
print('  ' + '-'*52)
for key, label in [("acc","Accuracy"),("prec","Precision"),("rec","Recall"),("f1","F1")]:
    a = m['A'][key]; b = m['B'][key]
    print(f'  {label:20} {a:>12.4f} {b:>12.4f}  {b-a:+.4f}')
for key, label in [("TP","TP"),("TN","TN"),("FP","FP"),("FN","FN")]:
    a = m['A'][key]; b = m['B'][key]
    print(f'  {label:20} {a:>12} {b:>12}  {b-a:+}')


BLOQUE B — Clasificación clip-level (umbral=5 frames)

  Procesando positivos (140 clips)...
    PAH1_C1_P1_V1_HB_3                        A=✅TP  B=❌FN
    PAH1_C1_P1_V1_HB_4                        A=✅TP  B=❌FN
    PAH1_C1_P2_V1_HB_1                        A=✅TP  B=✅TP
    PAH1_C1_P2_V1_HB_3                        A=✅TP  B=✅TP
    PAH1_C1_P4_V1_HB_1                        A=✅TP  B=❌FN
    PAH1_C1_P4_V1_HB_2                        A=✅TP  B=❌FN
    PAH1_C2_P3_V1_HB_1                        A=✅TP  B=✅TP
    PAH1_C2_P3_V1_HB_3                        A=✅TP  B=❌FN
    PAH1_C2_P3_V2_HB_2                        A=✅TP  B=❌FN
    PAH1_C2_P3_V2_HB_3                        A=✅TP  B=✅TP
    PAH1_C2_P5_V1_HB_1                        A=✅TP  B=✅TP
    PAH1_C2_P5_V1_HB_4                        A=✅TP  B=❌FN
    PAH1_C2_P5_V2_HB_3                        A=✅TP  B=❌FN
    PAH1_C2_P5_V2_HB_4                        A=✅TP  B=✅TP
    PAH2_C1_P1_V1_HB_1                        A=✅TP  B=❌FN
    PAH2_C1_P1_V1_HB_

In [ ]:
# ============================================================
# BLOQUE C — FP por categoría negativa
# ============================================================
print('\n' + '='*60)
print('BLOQUE C — Falsos positivos por categoría negativa')
print('='*60)

cat_stats = {cfg: defaultdict(lambda: {'total': 0, 'fp': 0}) for cfg in ['A', 'B']}

for cfg in ['A', 'B']:
    for r in clip_results[cfg]:
        if r['true'] != 0:
            continue
        cat = r['category']
        cat_stats[cfg][cat]['total'] += 1
        if r['pred'] == 1:
            cat_stats[cfg][cat]['fp'] += 1

print(f"\n  {'Cat':<5} {'Descripción':<24}  "
      f"{'FP_A':>5} {'%A':>6}  {'FP_B':>5} {'%B':>6}  {'Δ%':>7}")
print('  ' + '-'*68)

for cat in sorted(cat_stats['A'], key=lambda x: int(x[1:])):
    sA = cat_stats['A'][cat]
    sB = cat_stats['B'][cat]
    tot  = sA['total']
    pct_a = sA['fp'] / tot * 100 if tot > 0 else 0
    pct_b = sB['fp'] / tot * 100 if tot > 0 else 0
    desc  = cat_labels.get(cat, '')
    print(f"  {cat:<5} {desc:<24}  "
          f"{sA['fp']:>5} {pct_a:>6.1f}%  "
          f"{sB['fp']:>5} {pct_b:>6.1f}%  "
          f"{pct_b-pct_a:>+7.1f}pp")


BLOQUE C — Falsos positivos por categoría negativa

  Cat   Descripción                FP_A     %A   FP_B     %B       Δ%
  --------------------------------------------------------------------
  N1    Walking empty hands           3   33.3%      5   55.6%    +22.2pp
  N2    Jogging                       3   33.3%      3   33.3%     +0.0pp
  N3    Running                       0    0.0%      5   62.5%    +62.5pp
  N4    Sneaking empty hands          2   28.6%      1   14.3%    -14.3pp
  N5    Phone relaxed                 2   20.0%      5   50.0%    +30.0pp
  N6    Phone looking                 8   50.0%     10   62.5%    +12.5pp
  N7    Phone both hands              4   57.1%      1   14.3%    -42.9pp
  N8    Phone recording 1h            9   60.0%      4   26.7%    -33.3pp
  N9    Phone recording 2h            7   77.8%      4   44.4%    -33.3pp
  N10   Water bottle relaxed          2   22.2%      6   66.7%    +44.4pp
  N11   Drinking                      4   44.4%      6   66.7%    

In [ ]:
# ============================================================
# BLOQUE D — Guardar resultados en Drive
# ============================================================
print('\n' + '='*60)
print('BLOQUE D — Guardando resultados')
print('='*60)

for cfg in ['A', 'B']:
    label = 'sin_seg' if cfg == 'A' else 'con_seg'

    # --- TXT con resumen ---
    txt_path = Path(OUT_DIR) / f'results_config_{cfg}_{label}.txt'
    with open(txt_path, 'w') as f:
        f.write(f'=== CONFIG {cfg} — {label.upper()} ===\n\n')
        f.write(f'WEAPON_MODEL: {WEAPON_DET_WEIGHTS}\n')
        f.write(f'SEG_MODEL: {SEG_WEIGHTS}\n' if cfg == 'B' else 'SEG_MODEL: N/A\n')
        f.write(f'CONF_WEAPON={CONF_WEAPON} | IOU_NMS={IOU_NMS} | DETECTION_THRESHOLD={DETECTION_THRESHOLD}\n')
        if cfg == 'B':
            f.write(f'CONF_SEG={CONF_SEG} | Estrategia: máscara píxel a píxel\n')
        f.write('\n')

        f.write('--- BLOQUE A: mAP frame-level ---\n')
        f.write(f'Total GT boxes : {total_gt}\n')
        for key, label_m in [("mAP50","mAP@50"),("mAP5095","mAP@50:95"),
                              ("prec50","Precision@50"),("rec50","Recall@50")]:
            f.write(f'{label_m:15}: {map_metrics[cfg][key]:.4f}\n')
        f.write('\n')

        f.write('--- BLOQUE B: clip-level ---\n')
        for key, label_m in [("acc","Accuracy"),("prec","Precision"),
                              ("rec","Recall"),("f1","F1")]:
            f.write(f'{label_m:15}: {m[cfg][key]:.4f}\n')
        f.write(f'TP={m[cfg]["TP"]} TN={m[cfg]["TN"]} FP={m[cfg]["FP"]} FN={m[cfg]["FN"]}\n\n')

        f.write('--- BLOQUE C: FP por categoría negativa ---\n')
        for cat in sorted(cat_stats[cfg], key=lambda x: int(x[1:])):
            s = cat_stats[cfg][cat]
            tot = s['total']
            pct = s['fp'] / tot * 100 if tot > 0 else 0
            f.write(f'  {cat}: {s["fp"]}/{tot} ({pct:.1f}%)\n')

    print(f'  ✅ {txt_path}')

    # --- CSV detallado por clip ---
    csv_path = Path(OUT_DIR) / f'clip_results_{cfg}_{label}.csv'
    with open(csv_path, 'w', newline='') as f:
        fieldnames = ['clip','true','category','det_frames','total_frames','pred']
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(clip_results[cfg])
    print(f'  ✅ {csv_path}')

print('\n✅ Evaluación completa.')


BLOQUE D — Guardando resultados
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_A_sin_seg.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_A_sin_seg.csv
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/results_config_B_con_seg.txt
  ✅ /content/drive/MyDrive/TFM/experiments/video_tests/GAR/runs/seg_ablation/clip_results_B_con_seg.csv

✅ Evaluación completa.
